<div style="font-size: 24px; line-height: 1.6;">

# Misleading Variables: Some Columns Are Traps

## Cleanup is not housekeeping — it is deciding what evidence belongs

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced / reinforced: `columns`, `dtypes`, `astype`, `to_datetime`, `select_dtypes`, `corr`, `drop`.

**Concept learned: not every column deserves to survive EDA.**

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

A churn dataset contains useful predictors (plan, tenure), identifiers (`customer_id`), dates, and variables that *happen after* churn (`refund_after_churn`). The latter look powerfully predictive, but they leak the answer.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 1. Load the churn dataset

This file is an **HTML table** — the kind of thing you might scrape from a web page. `pd.read_html()` parses every `<table>` it finds and returns a **list of DataFrames**, so we take the first one with `[0]`.

</div>

In [2]:
churn = pd.read_html("../data/misleading_variables_churn.html")[0]
churn.head()

,customer_id,signup_date,age,tenure_months,plan,support_tickets,last_login_days_ago,refund_after_churn,churned
0,C00001,2020-11-22,70.0,12.1,free,0,6,False,False
1,C00002,2024-02-01,41.0,8.5,basic,1,6,False,False
2,C00003,2021-10-07,56.0,26.3,basic,3,14,False,False
3,C00004,2021-07-23,46.0,2.6,pro,0,4,False,False
4,C00005,2022-10-05,39.0,9.7,pro,0,4,False,False


In [3]:
churn.info()

<class 'pandas.DataFrame'>
RangeIndex: 3018 entries, 0 to 3017
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          3018 non-null   str    
 1   signup_date          3018 non-null   str    
 2   age                  2896 non-null   float64
 3   tenure_months        3018 non-null   float64
 4   plan                 3018 non-null   str    
 5   support_tickets      3018 non-null   int64  
 6   last_login_days_ago  3018 non-null   int64  
 7   refund_after_churn   3018 non-null   bool   
 8   churned              3018 non-null   bool   
dtypes: bool(2), float64(2), int64(2), str(3)
memory usage: 230.6 KB


<div style="font-size: 24px; line-height: 1.6;">

## 2. Column audit with `df.columns`

</div>

In [4]:
list(churn.columns)

['customer_id',
 'signup_date',
 'age',
 'tenure_months',
 'plan',
 'support_tickets',
 'last_login_days_ago',
 'refund_after_churn',
 'churned']

<div style="font-size: 24px; line-height: 1.6;">

## 3. Type audit with `df.dtypes`

</div>

In [5]:
churn.dtypes

customer_id                str
signup_date                str
age                    float64
tenure_months          float64
plan                       str
support_tickets          int64
last_login_days_ago      int64
refund_after_churn        bool
churned                   bool
dtype: object

<div style="font-size: 24px; line-height: 1.6;">

## 4. Convert with `df.astype()`

</div>

In [6]:
churn["churned"] = churn["churned"].astype("bool")
churn["churned"].dtype

dtype('bool')

<div style="font-size: 24px; line-height: 1.6;">

## 5. Convert dates with `pd.to_datetime()`

</div>

In [7]:
churn["signup_date"] = pd.to_datetime(churn["signup_date"])
churn["signup_date"].dtype

dtype('<M8[us]')

<div style="font-size: 24px; line-height: 1.6;">

## 6. Find suspicious correlations

</div>

In [8]:
numeric = churn.select_dtypes(include="number")
numeric.corr(numeric_only=True).round(3)

,age,tenure_months,support_tickets,last_login_days_ago
age,1.000,-0.003,0.052,0.145
tenure_months,-0.003,1.000,-0.015,-0.131
support_tickets,0.052,-0.015,1.000,0.153
last_login_days_ago,0.145,-0.131,0.153,1.000


<div style="font-size: 24px; line-height: 1.6;">

Pay special attention to anything that correlates strongly with `churned`. Ask: *could this value have been known at decision time, or is it a consequence of churn?*

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 7. Cross-check a suspect with the outcome

</div>

In [9]:
pd.crosstab(churn["churned"], churn["refund_after_churn"])

refund_after_churn,False,True
churned,,
False,2547,0
True,304,167


<div style="font-size: 24px; line-height: 1.6;">

If refunds only happen *after* churn, this column can perfectly predict the outcome — but only because the outcome already happened.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 8. Drop columns with `df.drop()`

</div>

In [10]:
safe = churn.drop(columns=["customer_id", "refund_after_churn", "last_login_days_ago"])
list(safe.columns)

['signup_date', 'age', 'tenure_months', 'plan', 'support_tickets', 'churned']

<div style="font-size: 24px; line-height: 1.6;">

Why each drop:

- `customer_id` — identifier, no predictive content
- `refund_after_churn` — happens after the outcome (**leak**)
- `last_login_days_ago` — measured at extraction time, may also leak

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 9. Drop rows vs. drop columns

</div>

In [11]:
print("Drop rows with any NA:  ", churn.dropna().shape)
print("Drop one column:        ", churn.drop(columns=["customer_id"]).shape)

Drop rows with any NA:   (2896, 9)
Drop one column:         (3018, 8)


<div style="font-size: 24px; line-height: 1.6;">

## Discussion

- For each remaining column, when in the customer's lifetime is its value known? Before churn, at churn, or after?
- Which columns would you keep for an honest churn-prediction EDA, and which would you justify dropping in writing?

</div>